# SPY buy & hold vs overnight liquidity-volatility gate

Replicates backtest da9d805cdf8097970633ce4ad6e7579e from daily bars: gate = dollar volume >= trailing 63-session median AND vol21 <= trailing 252-session median of vol21, decided at 15:30 ET using bars through the previous session (no look-ahead). Traded nights: close t -> open t+1, flat otherwise. Validation checks compare against the backtest log values.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

qb = QuantBook()
spy = qb.add_equity("SPY", Resolution.DAILY).symbol

# Daily bars from 2008-06 so the 63d and 252d gate windows are full by 2010-01.
hist = qb.history(spy, datetime(2008, 6, 1), datetime(2026, 7, 10), Resolution.DAILY)
if isinstance(hist.index, pd.MultiIndex):
    hist = hist.loc[spy]
df = hist[["open", "close", "volume"]].copy()

idx = pd.to_datetime(df.index)
if idx.tz is not None:
    idx = idx.tz_convert("America/New_York")
df.index = idx.normalize()

print(df.index[0].date(), "->", df.index[-1].date(), "| bars:", len(df))

In [ ]:
# Gate replication: dollar volume >= trailing 63d median AND vol21 <= trailing 252d median.
dv = df["close"] * df["volume"]
liq_ok = dv >= dv.rolling(63, min_periods=63).median()
vol21 = df["close"].pct_change().rolling(21, min_periods=21).std() * math.sqrt(252)
vol_ok = vol21 <= vol21.rolling(252, min_periods=252).median()

# Decision at 15:30 ET on day t uses bars through t-1 -> shift(1).
gate_night = (liq_ok & vol_ok).shift(1, fill_value=False)

# Frictionless nightly leg: close t -> open t+1; flat otherwise.
night_ret = df["open"].shift(-1) / df["close"] - 1.0
strat_full = night_ret * gate_night

t0 = pd.Timestamp("2010-01-04")
t_last_night = pd.Timestamp("2026-07-09")
t_last_bar = pd.Timestamp("2026-07-09")

strat_ret = strat_full.loc[t0:t_last_night].fillna(0.0)
gate_win = gate_night.loc[t0:t_last_night]
spy_ret = df["close"].pct_change().loc[t0:t_last_bar].fillna(0.0)

eq_strat = (1.0 + strat_ret).cumprod() * 100.0
eq_spy = (1.0 + spy_ret).cumprod() * 100.0

# Net-of-fee approximation from the backtest: $5,640.83 fees over 1,059 round trips.
fee_rt = 5640.83 / 1059.0 / 1000.0  # index units (100 = $100k)
eq_vals, eq = [], 100.0
for r, g in zip(strat_ret, gate_win):
    if g:
        eq = eq * (1.0 + r) - fee_rt
    eq_vals.append(eq)
eq_net = pd.Series(eq_vals, index=strat_ret.index)

print("nights traded:", int(gate_win.sum()), "(backtest: 1,059 - exact match)")
print(f"strategy total return (frictionless, dividends included): {eq_strat.iloc[-1] - 100:.2f}%")
print("  vs backtest fill-based log +30.98%: the log uses raw fill prices, which exclude")
print("  dividend cash on ex-dividend nights; adjusted research data includes them.")
print(f"strategy net of fees (approx): {eq_net.iloc[-1] - 100:.2f}% (backtest: +23.72%, same effect)")
print(f"SPY buy & hold (dividends included): {eq_spy.iloc[-1] - 100:.2f}%")
print("  backtest log +793.03% ends 2026-07-10 (~-1.4% day); research data ends 2026-07-09.")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(eq_spy.index, eq_spy.values, label="SPY buy & hold", color="tab:blue", lw=1.3)
ax.plot(eq_strat.index, eq_strat.values, label="Overnight gate (frictionless)", color="tab:orange", lw=1.3)
ax.plot(eq_net.index, eq_net.values, label="Overnight gate (net of fees, approx)", color="tab:green", lw=1.0, ls="--")
ax.set_yscale("log")
ax.set_title("SPY buy & hold vs overnight liquidity-volatility gate (indexed to 100, log scale)")
ax.set_ylabel("Equity (start = 100)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
win = 252
rs_strat = strat_ret.rolling(win, min_periods=win).mean() / strat_ret.rolling(win, min_periods=win).std() * math.sqrt(252)
rs_spy = spy_ret.rolling(win, min_periods=win).mean() / spy_ret.rolling(win, min_periods=win).std() * math.sqrt(252)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(rs_spy.index, rs_spy.values, label="SPY buy & hold", color="tab:blue", lw=1.2)
ax.plot(rs_strat.index, rs_strat.values, label="Overnight gate strategy", color="tab:orange", lw=1.2)
ax.axhline(0, color="gray", lw=0.8)
ax.set_title("Rolling 252-day Sharpe of daily returns (rf = 0)")
ax.set_ylabel("Sharpe")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly returns of the strategy (frictionless), year x month heatmap.
m = (1.0 + strat_ret).groupby([strat_ret.index.year, strat_ret.index.month]).prod() - 1.0
heat = m.unstack()

fig, ax = plt.subplots(figsize=(13, 7))
vmax = float(np.nanmax(np.abs(heat.values)))
im = ax.imshow(heat.values, cmap="RdYlGn", aspect="auto", vmin=-vmax, vmax=vmax)
ax.set_xticks(range(12))
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.1%}", ha="center", va="center", fontsize=8)
ax.set_title("Overnight gate strategy - monthly returns (frictionless)")
fig.colorbar(im, ax=ax, format="%.1f%%")
plt.tight_layout()
plt.show()